# Produce reusable jetAnalysis histograms

This is the single explicit production step for reusable histogram inputs. It resolves the configured data, official-MC, and standalone sources; projects every available nominal and systematic dijet eta TH2 in CM/Lab, Forward, and Backward frames in both shared standard and test $p_T^{ave}$ bin sets; and writes dedicated ROOT files beneath `hist_analysis/root_files/produced_histograms/`. Each output also retains the source histogram objects without modifying their keys, dimensions, binning, contents, or errors, including every official-MC unfolding family.

The EPPS21 theory artifact is validated in place because it has its own fixed set of available theory intervals. Downstream notebooks prefer these produced files and retain the original-source fallback. Stored eta projections are reused where available; unfolding and custom ranges may project the retained TH2/THnSparse inputs in memory.

Projection names encode the physical **per-jet** acceptance: source index 0 maps to `etaCut1p5`, index 4 to `etaCut1p9`, and index 7 to `etaCut3p0`. The upstream index-named ROOT objects are copied unchanged. Production adds only the configured half-open pTave projection interval; it does not reapply the jet cut to dijet eta. Run IDs are not eta-cut indices. Schema 2 metadata records the complete index-to-value table.

Regenerate official/data sources whose titles disagree with the current cut table before running this notebook. The preflight rejects stale cut orders before any bundle is written. The standalone analyzer macros already use the updated eight-cut order, but their existing ROOT outputs still need rebuilding. All downstream `ETA_CUTS` settings now share `DIJET_ETA_CUTS`; older local tuples containing 2.5 have been removed.

In [ ]:
# Cell role: initialize the project environment and explicit production helpers.
from pathlib import Path
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'CMakeLists.txt').is_file() and (p / 'hist_analysis').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Start Jupyter from the jetAnalysis repository root')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root
ROOT = load_root(batch=True)
from hist_analysis.python.histogram_production import (
    produce_all, source_catalog, validate_epps21,
)
ROOT.TH1.AddDirectory(False)
from hist_analysis.config.histograms import DIJET_ETA_CUTS, STANDARD_DIJET_ETA_CUT_INDEX
from hist_analysis.python.histogram_production import audit_source_cuts


## Resolve and audit every source

All paths must exist before production begins. The catalog covers MB, Jet60, Jet80, and Jet100 for Pb-going, p-going, and combined data; Embedding and PYTHIA for all three orientations; official ForestMiniAOD CP5; and standalone CUEP8M1 and CP5.

In [ ]:
print('Upstream eta-cut index to per-jet acceptance:', dict(enumerate(DIJET_ETA_CUTS)))
# Cell role: display resolved source-to-output mappings and fail on missing input.
CATALOG = source_catalog()
for identifier, entry in CATALOG.items():
    if not entry['source'].is_file():
        raise FileNotFoundError(entry['source'])
    print(f"{identifier}: {entry['source']} -> {entry['output']}")
print(f'Resolved {len(CATALOG)} production sources')
for identifier, entry in CATALOG.items():
    audit = audit_source_cuts(entry)
    print(f"{identifier}: validated {audit['eta_histograms']} eta histograms")


## Produce dedicated ROOT files

Each run recreates the dedicated outputs deterministically. Projection intervals use the existing half-open convention, $[low, high)$. Stored TH1 contents and `Sumw2` errors are the raw weighted yields; no display normalization or F/B division is persisted.

In [ ]:
# Cell role: execute the complete, explicit histogram production.
PRODUCTION_RESULTS = produce_all(CATALOG.values())
for result in PRODUCTION_RESULTS:
    print(f"{result['output']}: {result['projected']} projections, {result['copied']} copied source objects")


## Validate outputs and the fixed EPPS21 artifact

Every produced file must exist and contain its JSON production metadata. EPPS21 is opened and inventoried without changing its established theory-bin content.

In [ ]:
# Cell role: verify output creation, metadata, and the existing EPPS21 ROOT file.
for result in PRODUCTION_RESULTS:
    output = ROOT.TFile.Open(str(result['output']), 'READ')
    if not output or output.IsZombie():
        raise OSError(f"Cannot reopen {result['output']}")
    metadata = output.Get('histogramProductionMetadata')
    if not metadata:
        raise KeyError(f"Missing production metadata in {result['output']}")
    output.Close()
epps21 = validate_epps21()
print(f"Validated {len(PRODUCTION_RESULTS)} produced files")
print(f"EPPS21: {epps21['path']} ({len(epps21['top_level_keys'])} top-level keys)")
